In [1]:
import pandas as pd

df = pd.read_csv("../data/netflix_titles.csv")

In [2]:
#CLEAN ORIGINAL DF DATA TO GO INTO DB TABLE TITLES

#df = df.rename(columns={'show_id': 'title_id'})

# Just make sure bad entries are just empty strings

df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['year_added'] = df['date_added'].dt.year
df['date_added'] = df['date_added'].dt.strftime('%Y-%m-%d')
#df['year_added'] = df['year_added'].fillna(0).astype(int)

# Long way, not needed


# df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce').dt.strftime('%Y-%m-%d').fillna('')
#prepare to extract year added.
#df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

#extract year added
#df['year_added'] = df['date_added'].dt.year.fillna(0).astype(int)

#return date_added back to a string
#df['date_added'] = (
#    pd.to_datetime(df['date_added'], errors='coerce')
#    .dt.strftime('%Y-%m-%d')
#)

#Deal with bad data
df['country'] = df['country'].fillna('Unknown')
df['rating'] = df['rating'].fillna('Unknown')
df['duration'] = df['duration'].fillna('Unknown')

In [3]:
#Create a primary key in the titles df to prepare for db

df = df.reset_index(drop=True)
df['title_id'] = (df.index + 1).astype(int)


In [4]:
# CREATE GENRES DF


#parse listed_in to separate out individual genres

#genres_series = (
#    df['listed_in']
#    .str.split(',')
#    .str.strip()
#)

#create genres_df
#unique_genres = genres_series.dropna().unique()

#genres_df = pd.DataFrame({
#    'genre_id': range(1,len(unique_genres) + 1),
#    'genre': unique_genres
#})

# Keep what we need
exploded = df[['title_id', 'listed_in']].copy()

# Split comma-separated genres into lists
exploded['genre'] = exploded['listed_in'].str.split(',')

# Turn each list into multiple rows
exploded = exploded.explode('genre')

# Clean whitespace
exploded['genre'] = exploded['genre'].str.strip()

unique_genres = exploded['genre'].dropna().unique()

genres_df = pd.DataFrame({
    'genre': unique_genres
})

#PREPARE KEYS SO GENRES ARE ALREADY SORTED

# Create surrogate key
genres_df = genres_df.sort_values('genre').reset_index(drop=True)
genres_df['genre_id'] = genres_df.index + 1

# Put columns in the right order
genres_df = genres_df[['genre_id', 'genre']]


In [5]:
#MAKE THE MANY-TO-MANY JUNCTION TABLE DF (TITLE_GENRE_DF)

title_genres_df = exploded.merge(
    genres_df,
    on='genre',
    how='left'
)[['title_id', 'genre_id']]

In [6]:
#fill titles_df (parent) with the columns we want

titles_df = df[[
    'title_id',
    'title',
    'type',
    'release_year',
    'rating',
    'country',
    'date_added',
    'year_added'
]].copy()

In [7]:
#GET THE RIGHT PATH

from sqlalchemy import create_engine, text
from pathlib import Path

BASE_DIR = Path.cwd().parent   # project root
DB_PATH = BASE_DIR / "sql" / "streaming.db"

#print("Using DB at:", DB_PATH)

# Start your engines Vroom Vroom
engine = create_engine(f"sqlite:///{DB_PATH}", future=True)

In [8]:
#CREATE TV AND MOVIE DF'S

#movies_df = df.loc[df['type'] == 'Movie', ['title_id']].copy()
#tv_df = df.loc[df['type'] == 'TV Show', ['title_id']].copy()

#tv_df = df.loc[df['type'] == 'TV Show', ['title_id', 'duration']].copy()
#movies_df = df.loc[df['type'] == 'Movie', ['title_id', 'duration']].copy()

In [9]:
#FILL TV AND MOVIES DF'S

tv_df = df.query("type == 'TV Show'")[["title_id", "duration"]].copy()
tv_df['seasons'] = (
    tv_df['duration']
    .str.extract(r'^(\d+)\s+Season')
    .astype("Int64")
)
tv_df = tv_df[['title_id', 'seasons']]



movies_df = df.query("type == 'Movie'")[["title_id", "duration"]].copy()
movies_df['runtime_minutes'] = (
    movies_df['duration']
    .str.extract(r'(\d+)\s*min[s]?')
    .astype("Int64")
)
movies_df = movies_df[['title_id', 'runtime_minutes']]


#print(tv_df['duration'].head(20))


In [10]:
# FRESH START WITH STREAMING.DB

with engine.connect() as conn:
    conn.exec_driver_sql("PRAGMA foreign_keys = OFF;")
    conn.execute(text("DROP TABLE IF EXISTS movies;"))
    conn.execute(text("DROP TABLE IF EXISTS tv_shows;"))
    conn.execute(text("DROP TABLE IF EXISTS genres;"))
    conn.execute(text("DROP TABLE IF EXISTS title_genres;"))
    conn.execute(text("DROP TABLE IF EXISTS titles;"))
    conn.exec_driver_sql("PRAGMA foreign_keys = ON;")

#import os

#if os.path.exists("streaming.db"):
#    os.remove("streaming.db")

#if DB_PATH.exists():
#    DB_PATH.unlink()


from sqlalchemy import create_engine, text


In [11]:
#SETUP COLUMN NAMES FOR TITLES

with engine.connect() as conn:
    conn.execute(text("""
    CREATE TABLE titles(
        title_id INTEGER PRIMARY KEY, 
        title TEXT, 
        type TEXT, 
        release_year INTEGER, 
        rating TEXT, 
        country TEXT,
        date_added TEXT,
        year_added INTEGER,
        listed_in TEXT
    );
    """))

In [12]:
#Recreate movies and tv_shows child tables

with engine.connect() as conn:
    conn.execute(text("""
    CREATE TABLE movies (
        title_id INTEGER PRIMARY KEY,
        runtime_minutes INTEGER,
        FOREIGN KEY (title_id) REFERENCES titles(title_id)
    );
    """))

with engine.connect() as conn:
    conn.execute(text("""
    CREATE TABLE tv_shows (
        title_id INTEGER PRIMARY KEY,
        seasons INTEGER,
        FOREIGN KEY (title_id) REFERENCES titles(title_id)
    );
    """))

In [13]:
#create genre tables - many to many

with engine.connect() as conn:
    conn.execute(text("""
    CREATE TABLE genres (
        genre_id INTEGER PRIMARY KEY,
        genre TEXT UNIQUE
    );
    """))

with engine.connect() as conn:
    conn.execute(text("""
    CREATE TABLE title_genres (
        title_id INTEGER,
        genre_id INTEGER,
        FOREIGN KEY (title_id) REFERENCES titles(title_id),
        FOREIGN KEY (genre_id) REFERENCES genres(genre_id)
    );
    """))


In [14]:
titles_df.to_sql("titles", engine, if_exists="append", index=False)
movies_df.to_sql("movies", engine, if_exists="append", index=False)
tv_df.to_sql("tv_shows", engine, if_exists="append", index=False)
genres_df.to_sql("genres", engine, if_exists="append", index=False)
title_genres_df.to_sql("title_genres", engine, if_exists="append", index=False)

19323

In [15]:
# Save cleaned data to csv files

titles_df.to_csv("../exports/streaming_cleaned.csv", index=False)
movies_df.to_csv("../exports/movies_cleaned.csv", index=False)
tv_df.to_csv("../exports/tv_cleaned.csv", index=False)
genres_df.to_csv("../exports/genres_cleaned.csv", index=False)
title_genres_df.to_csv("../exports/title_genres_cleaned.csv", index=False)